<a href="https://colab.research.google.com/github/anupbagale/Whole-Brain-Segmentation/blob/main/notebooks/01_data_and_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data and labels

This notebook looks at the OASIS-3 scans before any training: how many scans there are, which of the
44 FreeSurfer structures appear in them, how big each structure is, and what the two label sets look like.
It runs on CPU.

In [ ]:
import os, sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("wbseg"):
        !git clone -q https://github.com/anupbagale/Whole-Brain-Segmentation.git
        %cd Whole-Brain-Segmentation
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
# Change these to where the data is on your Drive
IMAGE_DIR = "/content/drive/MyDrive/OASIS3_Dataset_Part4/Image_Folder"   # T1 images (.nii.gz)
LABEL_DIR = "/content/drive/MyDrive/OASIS3_Dataset_Part4/Label_Folder"   # FreeSurfer aseg label maps
LUT_PATH = "/content/drive/MyDrive/FreeSurferColorLUT.txt"               # optional, only for colours
RUN_DIR = "/content/drive/MyDrive/whole_brain_seg_runs"                   # checkpoints and predictions

if not os.path.exists(LUT_PATH):
    LUT_PATH = None
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wbseg.data import find_pairs
from wbseg.labels import FULL_LABELS, REDUCED_LABELS, NAMES, colour_table
from wbseg.volumes import label_volumes
from wbseg import plots

## Scans

In [ ]:
pairs = find_pairs(IMAGE_DIR, LABEL_DIR)
print(len(pairs), "image / label pairs")
img = nib.load(pairs[0]["image"])
print("shape", img.shape, " voxel size (mm)", img.header.get_zooms()[:3])

## The two label sets

The full set has every aseg structure present in the data (44 structures + background = 45 classes).
The reduced set keeps 27 structures and moves the small or rare ones (amygdala, accumbens, inferior
lateral ventricles, vessels, CSF, white matter hypointensities, optic chiasm and the five corpus callosum
segments) to background.

In [ ]:
dropped = [l for l in FULL_LABELS[1:] if l not in REDUCED_LABELS]
print("classes:", len(FULL_LABELS), "full,", len(REDUCED_LABELS), "reduced")
print("left out of the reduced set:")
for l in dropped:
    print(f"  {l:4d}  {NAMES[l]}")

## Structure sizes

Volume of every structure in every scan, straight from the FreeSurfer label maps. This shows how unbalanced
the problem is: the cerebral white matter is thousands of times bigger than the optic chiasm or the vessel labels.

In [ ]:
rows = []
for p in pairs:
    for lab, vol in label_volumes(p["label"], FULL_LABELS).items():
        rows.append({"id": p["id"], "label": lab, "structure": NAMES[lab], "volume_ml": vol})
vols = pd.DataFrame(rows)

sizes = (vols.groupby(["label", "structure"])["volume_ml"]
         .agg(median_ml="median", present_in=lambda v: int((v > 0).sum()))
         .reset_index().sort_values("median_ml", ascending=False))
sizes["in_reduced_set"] = sizes["label"].isin(REDUCED_LABELS)
sizes.to_csv("results/structure_sizes.csv", index=False)
sizes

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
s = sizes.sort_values("median_ml")
colour = np.where(s["in_reduced_set"], plots.BLUE, plots.ORANGE)
ax.barh(s["structure"], s["median_ml"], color=colour)
ax.set_xscale("log")
ax.set_xlabel("Median volume (ml, log scale)")
ax.tick_params(axis="y", labelsize=8)
ax.set_title("Blue: in both label sets. Orange: full set only.", loc="left", fontsize=10)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
fig.savefig("figures/structure_sizes.png", dpi=150)
plt.show()

## Example scan with both label sets

In [ ]:
img = nib.as_closest_canonical(nib.load(pairs[0]["image"]))
lab = nib.as_closest_canonical(nib.load(pairs[0]["label"]))
image, label = img.get_fdata(), np.asarray(lab.dataobj).astype(int)
reduced = np.where(np.isin(label, REDUCED_LABELS), label, 0)

box = plots.brain_box(label, axis=1, index=128)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
plots.overlay(image, label, FULL_LABELS, colour_table(FULL_LABELS), axis=1, index=128, ax=axes[0], box=box, alpha=0)
axes[0].set_title("T1", loc="left")
plots.overlay(image, label, FULL_LABELS, colour_table(FULL_LABELS, LUT_PATH), axis=1, index=128, ax=axes[1], box=box)
axes[1].set_title("45 classes", loc="left")
plots.overlay(image, reduced, REDUCED_LABELS, colour_table(REDUCED_LABELS, LUT_PATH), axis=1, index=128, ax=axes[2], box=box)
axes[2].set_title("28 classes", loc="left")
fig.tight_layout()
fig.savefig("figures/label_sets.png", dpi=150)
plt.show()